# 04 베이스라인: core 13개 × (로지스틱 회귀, RBF-SVM)

- 평가: `src/evaluate.py`의 중첩 CV (바깥 5-fold × 안쪽 4-fold, seed 5개)
- 비교 1: 단순 모델(로지스틱 회귀) vs 비선형 모델(RBF-SVM)
- 비교 2: 정직한 점수 vs 누수 버전 점수 (임계값을 점수 낸 데이터로 고르면 얼마나 부풀려지나)
- 바닥 점수: 전부 고장 0.556, RMS 관리도 0.080 (01 노트북)

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import fbeta_score

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.data import load_dataset
from src.features import feature_table
from src.evaluate import nested_cv, best_f2_threshold

X, y, names = load_dataset(ROOT / "data" / "TeamProject_train_변경")
F = feature_table(X).to_numpy()
groups = np.arange(len(y))  # 파일 하나 = 그룹 하나 (나중 슬라이싱 대비)

# 호출할 때마다 새 모델을 만든다 → fold마다 처음부터 학습
MODELS = {
    "로지스틱 회귀": lambda: make_pipeline(RobustScaler(), LogisticRegression(class_weight="balanced", max_iter=5000)),
    "RBF-SVM": lambda: make_pipeline(RobustScaler(), SVC(kernel="rbf", class_weight="balanced")),
}

## 1. 정직한 점수 (중첩 CV)

In [2]:
results = {name: nested_cv(F, y, groups, make) for name, make in MODELS.items()}

rows = []
for name, r in results.items():
    rows.append({
        "모델": name,
        "F2 평균": r["f2"].mean(), "F2 표준편차": r["f2"].std(),
        "재현율": r["recall"].mean(), "정밀도": r["precision"].mean(),
        "놓친 고장(평균)": r["miss_counts"].sum() / len(r["f2"]),
        "오경보(평균)": r["fp_counts"].sum() / len(r["f2"]),
    })
print(pd.DataFrame(rows).round(3).to_string(index=False))
for name, r in results.items():
    print(f"\n{name} seed별 F2:", r["f2"].round(3))

     모델  F2 평균  F2 표준편차   재현율   정밀도  놓친 고장(평균)  오경보(평균)
로지스틱 회귀  0.608    0.055 0.753 0.348        7.4     43.4
RBF-SVM  0.606    0.026 0.773 0.326        6.8     48.0

로지스틱 회귀 seed별 F2: [0.65  0.505 0.653 0.638 0.596]

RBF-SVM seed별 F2: [0.619 0.619 0.591 0.638 0.565]


## 2. 자주 틀리는 파일 (seed 5개 중 3번 이상)

In [3]:
names_arr = np.array(names)
for name, r in results.items():
    print(f"{name}")
    print("  놓친 고장:", names_arr[r["miss_counts"] >= 3].tolist())
    print("  오경보 정상:", names_arr[r["fp_counts"] >= 3].tolist())

로지스틱 회귀
  놓친 고장: ['abn_002', 'abn_005', 'abn_013', 'abn_014', 'abn_015', 'abn_022', 'abn_023', 'abn_029']
  오경보 정상: ['nor_004', 'nor_006', 'nor_008', 'nor_010', 'nor_011', 'nor_013', 'nor_015', 'nor_022', 'nor_023', 'nor_025', 'nor_027', 'nor_032', 'nor_035', 'nor_037', 'nor_038', 'nor_039', 'nor_040', 'nor_045', 'nor_046', 'nor_048', 'nor_051', 'nor_052', 'nor_054', 'nor_055', 'nor_056', 'nor_066', 'nor_068', 'nor_072', 'nor_079', 'nor_086', 'nor_088', 'nor_089', 'nor_090', 'nor_093', 'nor_095', 'nor_106', 'nor_107', 'nor_116', 'nor_119']
RBF-SVM
  놓친 고장: ['abn_002', 'abn_003', 'abn_020', 'abn_022', 'abn_023', 'abn_025', 'abn_029']
  오경보 정상: ['nor_001', 'nor_004', 'nor_006', 'nor_009', 'nor_015', 'nor_016', 'nor_022', 'nor_023', 'nor_025', 'nor_027', 'nor_028', 'nor_030', 'nor_032', 'nor_033', 'nor_035', 'nor_037', 'nor_038', 'nor_040', 'nor_045', 'nor_046', 'nor_048', 'nor_050', 'nor_054', 'nor_055', 'nor_057', 'nor_060', 'nor_065', 'nor_066', 'nor_068', 'nor_070', 'nor_072', 'nor_07

## 3. 누수 버전과 비교

누수 버전: 바깥 CV로 OOF 점수를 만든 뒤, **같은 OOF 점수**로 임계값을 고르고 그 OOF로 F₂를 계산한다.
임계값이 평가 파일의 정답을 보고 정해지므로 점수가 부풀려진다. 정직한 점수보다 높게 나와야 정상이다.

In [4]:
def leaky_f2(make_model, seeds=range(5), n_splits=5):
    f2s = []
    for seed in seeds:
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        oof = np.empty(len(y))
        for tr, te in cv.split(F, y, groups):
            m = make_model().fit(F[tr], y[tr])
            oof[te] = m.decision_function(F[te])
        t = best_f2_threshold(y, oof)              # 평가할 데이터의 정답으로 임계값 선택 (누수)
        f2s.append(fbeta_score(y, (oof >= t).astype(int), beta=2))
    return np.array(f2s)

rows = []
for name, make in MODELS.items():
    leak = leaky_f2(make)
    rows.append({"모델": name, "정직한 F2": results[name]["f2"].mean(), "누수 F2": leak.mean(),
                 "차이": leak.mean() - results[name]["f2"].mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

     모델  정직한 F2  누수 F2   차이
로지스틱 회귀   0.608  0.669 0.06
RBF-SVM   0.606  0.656 0.05


## 4. 오류 분석: 두 모델이 모두 자주 놓친 고장

각 피처값이 정상 120개 중 몇 번째 백분위에 있는지 본다. 50 근처면 정상과 구분되지 않는 값, 95 이상·5 이하면 정상 범위 밖이다.

In [5]:
cols = feature_table(X[:1]).columns
both_missed = (results["로지스틱 회귀"]["miss_counts"] >= 3) & (results["RBF-SVM"]["miss_counts"] >= 3)
F0 = F[y == 0]
pct = pd.DataFrame(
    [[(F0[:, j] < F[i, j]).mean() * 100 for j in range(F.shape[1])] for i in np.where(both_missed)[0]],
    index=names_arr[both_missed], columns=cols,
)
print("놓친 고장의 정상 대비 백분위 (0~100)")
print(pct.round(0).astype(int).T.to_string())

# 비교: 잘 잡힌 고장(두 모델 모두 놓친 적 없음)의 백분위 중앙값
caught = (y == 1) & (results["로지스틱 회귀"]["miss_counts"] == 0) & (results["RBF-SVM"]["miss_counts"] == 0)
pct_caught = [[(F0[:, j] < F[i, j]).mean() * 100 for j in range(F.shape[1])] for i in np.where(caught)[0]]
print(f"\n잘 잡힌 고장 {caught.sum()}개의 백분위 중앙값")
print(pd.Series(np.median(pct_caught, axis=0), index=cols).round(0).astype(int).to_string())

놓친 고장의 정상 대비 백분위 (0~100)
                         abn_002  abn_022  abn_023  abn_029
T01_rms                       28       57       53       20
T02_peak                      20       53       85       29
T07_kurtosis                  11       28       89       50
T08_skewness                  40       92       98       16
T15_hjorth_complexity         38       88       72       48
T16_tkeo_max                  30       38       60       45
W01_win_rms_max_to_mean       23       53       92       67
F01_spectral_centroid         54       19       18       83
F07_peak1_freq                12       32       76       24
F08_peak1_amp                 29       72       58       70
B01_band_low_ratio            31       86       66       44
B02_band_mid_ratio            83       29       56       42
B03_band_high_ratio           54        9       22       63

잘 잡힌 고장 14개의 백분위 중앙값
T01_rms                    92
T02_peak                   95
T07_kurtosis               88
T08_skewness           